# Duplicate Invoice Detection Notebook

This notebook detects suspected duplicate vendor invoices in simulated SAP invoice verification data. The analysis mirrors SAP MIRO duplicate-check concepts by comparing vendor, invoice reference, invoice amount, and posting/document timing.

## 1. Import and load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.options.display.float_format = "{:,.0f}".format

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "03_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "03_Data"
OUTPUT_DIR = PROJECT_ROOT / "05_Python_Analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def yen(value):
    return f"¥{value:,.0f}"

invoices = pd.read_csv(DATA_DIR / "invoices.csv", parse_dates=["invoice_date", "posting_date", "payment_date"])
vendors = pd.read_csv(DATA_DIR / "vendors.csv", parse_dates=["registration_date"])

invoice_data = invoices.merge(vendors[["vendor_id", "vendor_name", "vendor_category", "is_ghost_vendor"]], on="vendor_id", how="left")
invoice_data.head()


## 2. Statistical overview of invoices

In [ ]:
invoice_overview = pd.DataFrame({
    "metric": [
        "invoice_count",
        "unique_vendors",
        "total_invoice_value_jpy",
        "average_invoice_value_jpy",
        "paid_invoice_value_jpy",
        "blocked_invoice_count",
    ],
    "value": [
        len(invoice_data),
        invoice_data["vendor_id"].nunique(),
        invoice_data["invoice_amount"].sum(),
        invoice_data["invoice_amount"].mean(),
        invoice_data.loc[invoice_data["payment_status"].eq("Paid"), "invoice_amount"].sum(),
        invoice_data["payment_status"].eq("Blocked").sum(),
    ],
})
invoice_overview


In [ ]:
invoice_data[["invoice_amount"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])


## 3. Duplicate detection algorithm

Two criteria are used:

1. **Exact duplicate candidates**: same vendor and same invoice number, or same vendor, same amount, and same posting date.
2. **Fuzzy duplicate candidates**: same vendor, invoice amount within 10%, and posting dates within 30 days.

Posting date is used for the timing window because SAP duplicate-payment exposure is created when documents are posted for payment processing. The fuzzy rule intentionally catches cases where the invoice number was changed to evade a simple SAP reference-number duplicate check.


In [ ]:
exact_by_reference = invoice_data[invoice_data.duplicated(["vendor_id", "invoice_number"], keep=False)].copy()
exact_by_reference["duplicate_rule"] = "Exact: same vendor + invoice number"

exact_by_amount_posting_date = invoice_data[invoice_data.duplicated(["vendor_id", "invoice_amount", "posting_date"], keep=False)].copy()
exact_by_amount_posting_date["duplicate_rule"] = "Exact: same vendor + amount + posting date"

# Pair each vendor's invoices to find near-duplicates within SAP-style posting review windows.
pairs = invoice_data.merge(invoice_data, on="vendor_id", suffixes=("_a", "_b"))
pairs = pairs[pairs["invoice_id_a"] < pairs["invoice_id_b"]].copy()
pairs["amount_diff_pct"] = (pairs["invoice_amount_a"] - pairs["invoice_amount_b"]).abs() / pairs[["invoice_amount_a", "invoice_amount_b"]].max(axis=1)
pairs["days_between_postings"] = (pairs["posting_date_a"] - pairs["posting_date_b"]).abs().dt.days

fuzzy_pairs = pairs[(pairs["amount_diff_pct"] <= 0.10) & (pairs["days_between_postings"] <= 30)].copy()
fuzzy_pairs["duplicate_rule"] = "Fuzzy: same vendor + within 10% amount + within 30 posting days"
fuzzy_pairs = fuzzy_pairs.sort_values(["vendor_id", "posting_date_a", "posting_date_b", "amount_diff_pct"])
fuzzy_pairs.head(20)


## 4. Results analysis

In [ ]:
exact_ids = set(exact_by_reference["invoice_id"]).union(exact_by_amount_posting_date["invoice_id"])
fuzzy_ids = set(fuzzy_pairs["invoice_id_a"]).union(fuzzy_pairs["invoice_id_b"])

suspected_duplicates = invoice_data[invoice_data["invoice_id"].isin(exact_ids | fuzzy_ids)].copy()
suspected_duplicates["detected_exact_duplicate"] = suspected_duplicates["invoice_id"].isin(exact_ids)
suspected_duplicates["detected_fuzzy_duplicate"] = suspected_duplicates["invoice_id"].isin(fuzzy_ids)
suspected_duplicates["risk_reason"] = suspected_duplicates.apply(
    lambda row: "Exact and fuzzy duplicate" if row["detected_exact_duplicate"] and row["detected_fuzzy_duplicate"]
    else "Exact duplicate" if row["detected_exact_duplicate"]
    else "Fuzzy duplicate",
    axis=1,
)

suspected_duplicates = suspected_duplicates.sort_values(["vendor_id", "invoice_date", "invoice_amount"])
suspected_duplicates.to_csv(OUTPUT_DIR / "duplicate_invoice_suspects.csv", index=False)
fuzzy_pairs.to_csv(OUTPUT_DIR / "duplicate_invoice_pairs.csv", index=False)

summary_by_rule = suspected_duplicates["risk_reason"].value_counts().rename_axis("risk_reason").reset_index(name="invoice_count")
summary_by_rule


In [ ]:
duplicate_vendor_summary = (
    suspected_duplicates.groupby(["vendor_id", "vendor_name"], as_index=False)
    .agg(
        suspected_invoice_count=("invoice_id", "count"),
        exposure_jpy=("invoice_amount", "sum"),
        paid_exposure_jpy=("invoice_amount", lambda s: s[suspected_duplicates.loc[s.index, "payment_status"].eq("Paid")].sum()),
        first_invoice_date=("invoice_date", "min"),
        last_invoice_date=("invoice_date", "max"),
    )
    .sort_values(["exposure_jpy", "suspected_invoice_count"], ascending=False)
)
duplicate_vendor_summary.to_csv(OUTPUT_DIR / "duplicate_invoice_vendor_summary.csv", index=False)
duplicate_vendor_summary.head(15)


## 5. Visualization of duplicates by vendor

In [ ]:
top_vendors = duplicate_vendor_summary.head(10).copy()
plt.figure(figsize=(12, 6))
sns.barplot(data=top_vendors, y="vendor_name", x="exposure_jpy", hue="suspected_invoice_count", dodge=False)
plt.title("Top Vendors by Suspected Duplicate Invoice Exposure")
plt.xlabel("Suspected exposure (JPY)")
plt.ylabel("Vendor")
plt.legend(title="Invoice count")
plt.tight_layout()
plt.show()


In [ ]:
timeline = (
    suspected_duplicates.assign(month=suspected_duplicates["invoice_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(suspected_invoice_count=("invoice_id", "count"), exposure_jpy=("invoice_amount", "sum"))
)

fig, ax1 = plt.subplots(figsize=(12, 5))
sns.lineplot(data=timeline, x="month", y="suspected_invoice_count", marker="o", ax=ax1, label="Count")
ax1.set_ylabel("Suspected invoice count")
ax2 = ax1.twinx()
sns.lineplot(data=timeline, x="month", y="exposure_jpy", marker="s", color="crimson", ax=ax2, label="Exposure")
ax2.set_ylabel("Exposure (JPY)")
ax1.set_title("Suspected Duplicate Invoice Timeline")
fig.tight_layout()
plt.show()


## 6. Risk quantification — total JPY at risk

In [ ]:
total_financial_exposure = suspected_duplicates["invoice_amount"].sum()
paid_financial_exposure = suspected_duplicates.loc[suspected_duplicates["payment_status"].eq("Paid"), "invoice_amount"].sum()

risk_quantification = pd.DataFrame({
    "risk_metric": ["suspected_duplicate_invoices", "vendors_impacted", "total_exposure_jpy", "paid_exposure_jpy"],
    "value": [len(suspected_duplicates), suspected_duplicates["vendor_id"].nunique(), total_financial_exposure, paid_financial_exposure],
})
risk_quantification


## 7. Recommendations

- Activate and monitor SAP MIRO duplicate invoice checks for vendor, invoice reference, company code, document date, and amount.
- Review vendors with repeated fuzzy duplicates because changed invoice numbers may evade exact RBKP-XBLNR checks.
- Block payment for open suspected duplicates until AP completes vendor confirmation.
- Use the exported suspect files in `05_Python_Analysis/outputs/` as audit workpapers for follow-up.